# Predictive Analytics Using Historical Data

This notebook walks through the full workflow interactively:
1. Load & explore historical data
2. Clean & preprocess
3. Engineer features
4. Train regression + time-series models
5. Evaluate accuracy
6. Visualize predictions & trends

> For the automated, script-based version of this pipeline see `main.py` in the project root.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data_generator import save_sample_data
from src.preprocessing import build_feature_dataset, time_based_split, load_data, handle_missing_values, handle_outliers
from src.models import train_linear_regression, train_random_forest, train_arima, get_feature_importance
from src.evaluate import evaluate_predictions, print_report

%matplotlib inline

## 1. Load historical data
Replace `data/sample_sales_data.csv` with your own dataset — just make sure it has a date column and a numeric target column.

In [ ]:
DATA_PATH = '../data/sample_sales_data.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.describe()

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['date'], df['sales'])
plt.title('Raw Historical Data (before cleaning)')
plt.show()

## 2. Clean, preprocess & engineer features

In [ ]:
features_df = build_feature_dataset(DATA_PATH, target_col='sales')
features_df.head()

## 3. Chronological train/test split

In [ ]:
train, test = time_based_split(features_df, test_size=0.15)

## 4. Train models

In [ ]:
lr_model, lr_preds, scaler = train_linear_regression(train, test, 'sales')
rf_model, rf_preds = train_random_forest(train, test, 'sales')
arima_model, arima_preds = train_arima(train, test, 'sales')

## 5. Evaluate & compare

In [ ]:
y_test = test['sales']
results = [
    evaluate_predictions(y_test, lr_preds, 'Linear Regression'),
    evaluate_predictions(y_test, rf_preds, 'Random Forest'),
    evaluate_predictions(y_test, arima_preds, 'ARIMA'),
]
comparison_df = print_report(results)
comparison_df

## 6. Visualize forecast vs actual

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(test.index, y_test, label='Actual', color='black', linewidth=2)
plt.plot(test.index, lr_preds, label='Linear Regression', linestyle='--')
plt.plot(test.index, rf_preds, label='Random Forest', linestyle='--')
plt.plot(test.index, arima_preds, label='ARIMA', linestyle='--')
plt.legend()
plt.title('Forecast vs Actual')
plt.show()

## 7. Feature importance (Random Forest)

In [ ]:
importance = get_feature_importance(rf_model)
importance.plot(kind='barh', figsize=(9,6), title='Feature Importance')
plt.gca().invert_yaxis()
plt.show()